# Notebook 2: What Is a Vector Database?

**Who this is for:** Digital Humanities students with no prior AI or programming background.

**What you will learn**
1. What a vector database is, and why we need one
2. How to store texts (English and Arabic) in a small vector database
3. How to search it by meaning, in either language
4. How to filter results (for example, Arabic texts only)
5. How to add new documents

**How to run this notebook:** run each cell in order (press the play button or `Shift+Enter`). No API key is needed. Everything runs free in Google Colab. Lines that start with `#` are comments: notes for humans that Python ignores.

## 0. Setup

In [ ]:
# Install the libraries. "-q" keeps the output short. This takes about a minute.
!pip install -q chromadb sentence-transformers matplotlib pandas

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display                        # shows tables nicely
import chromadb                                            # the vector database
from sentence_transformers import SentenceTransformer     # turns text into vectors (Notebook 1)

## 1. The idea

In Notebook 1 we compared a question with **every** document, one by one. That is fine for 14 documents. It becomes too slow for millions.

A **vector database** is built for this job. For every item it stores:
1. the **vector** (the embedding: numbers that represent the meaning),
2. the **original text**,
3. **metadata**: labels such as language, topic or date.

It can then find the closest vectors **without checking every one**. Think of a library catalogue organised by meaning instead of by title.

| | Ordinary database | Vector database |
|---|---|---|
| You search with | exact words, numbers, ids | a question or sentence |
| It finds | exact matches | the closest meaning |

Below is our small collection: 14 short passages about the history of books, learning and travel, each written in **English and Arabic**.

In [ ]:
#@title Load our collection of 14 historical passages (double-click to show the code) { display-mode: "form" }
PASSAGES = [
    {"id": "p01", "title": "House of Wisdom", "topic": "learning",
     "en": "The House of Wisdom in Baghdad was a major centre where scholars translated Greek, Persian and Indian texts into Arabic.",
     "ar": "كان بيت الحكمة في بغداد مركزًا كبيرًا ترجم فيه العلماء النصوص اليونانية والفارسية والهندية إلى العربية."},
    {"id": "p02", "title": "Gutenberg", "topic": "writing and books",
     "en": "Around 1450, Johannes Gutenberg introduced movable-type printing in Europe, which made books cheaper and faster to produce.",
     "ar": "حوالي عام 1450 أدخل يوهانس غوتنبرغ الطباعة بالحروف المتحركة في أوروبا، فأصبح إنتاج الكتب أرخص وأسرع."},
    {"id": "p03", "title": "Ibn Battuta", "topic": "travel and trade",
     "en": "In 1325 Ibn Battuta left Tangier for the pilgrimage to Mecca and went on to travel through Africa, Asia and Europe for nearly thirty years.",
     "ar": "في عام 1325 غادر ابن بطوطة طنجة لأداء الحج، ثم سافر في أفريقيا وآسيا وأوروبا قرابة ثلاثين عامًا."},
    {"id": "p04", "title": "Rosetta Stone", "topic": "language and literature",
     "en": "The Rosetta Stone carries one decree in three scripts, which helped Jean-François Champollion decipher Egyptian hieroglyphs in 1822.",
     "ar": "يحمل حجر رشيد مرسومًا واحدًا بثلاث كتابات، وقد ساعد جان فرانسوا شامبليون على فك رموز الهيروغليفية المصرية عام 1822."},
    {"id": "p05", "title": "Monks and scriptoria", "topic": "writing and books",
     "en": "In medieval Europe, monks copied manuscripts by hand in rooms called scriptoria.",
     "ar": "في أوروبا في العصور الوسطى، كان الرهبان ينسخون المخطوطات بأيديهم في قاعات خاصة تُسمى «سكريبتوريا»."},
    {"id": "p06", "title": "Digital archives", "topic": "archives and libraries",
     "en": "Digital archives let researchers search millions of scanned newspaper pages from their own computers.",
     "ar": "تتيح الأرشيفات الرقمية للباحثين البحث في ملايين الصفحات الصحفية الممسوحة ضوئيًا من حواسيبهم."},
    {"id": "p07", "title": "Silk Road", "topic": "travel and trade",
     "en": "The Silk Road was a network of trade routes that connected China with the Mediterranean world.",
     "ar": "كان طريق الحرير شبكة من طرق التجارة ربطت الصين بعالم البحر المتوسط."},
    {"id": "p08", "title": "Ibn Khaldun", "topic": "learning",
     "en": "Ibn Khaldun was born in Tunis in 1332 and wrote the Muqaddimah, an introduction to the study of history, in 1377.",
     "ar": "وُلد ابن خلدون في تونس عام 1332، وكتب «المقدمة»، وهي تمهيد لدراسة التاريخ، عام 1377."},
    {"id": "p09", "title": "Al-Azhar", "topic": "learning",
     "en": "Al-Azhar in Cairo was founded in 970 and became one of the most important centres of Islamic learning.",
     "ar": "تأسس الأزهر في القاهرة عام 970، وأصبح من أهم مراكز العلم الإسلامي."},
    {"id": "p10", "title": "Al-Qarawiyyin", "topic": "learning",
     "en": "The University of al-Qarawiyyin in Fez was founded in 859 by Fatima al-Fihri.",
     "ar": "أسست فاطمة الفهرية جامعة القرويين في فاس عام 859."},
    {"id": "p11", "title": "Library of Alexandria", "topic": "archives and libraries",
     "en": "The Library of Alexandria, founded in the third century BCE, was one of the largest libraries of the ancient world.",
     "ar": "كانت مكتبة الإسكندرية، التي تأسست في القرن الثالث قبل الميلاد، من أكبر مكتبات العالم القديم."},
    {"id": "p12", "title": "Papermaking", "topic": "writing and books",
     "en": "From the eighth century, papermaking spread across the Islamic world, which made books cheaper and helped libraries grow.",
     "ar": "منذ القرن الثامن انتشرت صناعة الورق في العالم الإسلامي، فأصبحت الكتب أرخص ونمت المكتبات."},
    {"id": "p13", "title": "Al-Mutanabbi", "topic": "language and literature",
     "en": "Al-Mutanabbi, a poet of the tenth century, is considered one of the greatest poets of the Arabic language.",
     "ar": "يُعد المتنبي، شاعر القرن العاشر، من أعظم شعراء اللغة العربية."},
    {"id": "p14", "title": "Bulaq Press", "topic": "writing and books",
     "en": "Around 1820 the Bulaq Press in Cairo began printing books in Arabic on a large scale.",
     "ar": "حوالي عام 1820 بدأت مطبعة بولاق في القاهرة طباعة الكتب العربية على نطاق واسع."},
]

In [ ]:
# Show what we loaded: one line per passage.
for p in PASSAGES:
    print(p["id"], "-", p["title"], f"({p['topic']})")

## 2. Step 1: turn the texts into vectors

We use the same multilingual model as in Notebook 1. It understands English and Arabic.

In [ ]:
# The embedding model (the first run downloads about 450 MB).
embedder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# Every passage is stored twice: once in English and once in Arabic.
ids, texts, metas = [], [], []
for p in PASSAGES:
    for lang in ["en", "ar"]:
        ids.append(f"{p['id']}_{lang}")                    # a unique name for each item
        texts.append(p[lang])                              # the original text
        metas.append({"title": p["title"], "topic": p["topic"], "language": lang})   # the labels

vectors = embedder.encode(texts).tolist()                  # one vector per text
print(len(texts), "texts embedded, each with", len(vectors[0]), "numbers")

## 3. Step 2: store them in the database

In [ ]:
# Create an empty database that lives in the computer's memory.
db = chromadb.EphemeralClient()

# If you run this cell twice, delete the old collection first so that the name is free.
try:
    db.delete_collection("heritage")
except Exception:
    pass

# A "collection" is like a table. "cosine" means: compare vectors by their angle (Notebook 1).
collection = db.create_collection(name="heritage", metadata={"hnsw:space": "cosine"})

# Store, for every item: its id, its vector, its original text and its labels.
collection.add(ids=ids, embeddings=vectors, documents=texts, metadatas=metas)
print("Items stored:", collection.count())

## 4. Step 3: search by meaning

The function `search()` embeds your question and asks the database for the closest items. A `similarity` of 1 means the same meaning; near 0 means unrelated.

In [ ]:
def search(query, k=3, where=None):
    """Find the k stored texts closest in meaning to `query`.
    `where` is an optional filter on the labels (metadata)."""
    query_vector = embedder.encode(query).tolist()                  # embed the question
    options = dict(query_embeddings=[query_vector], n_results=k)
    if where:
        options["where"] = where
    found = collection.query(**options)                             # ask the database

    rows = []
    for i in range(len(found["ids"][0])):
        meta = found["metadatas"][0][i]
        rows.append({
            "similarity": round(1 - found["distances"][0][i], 2),   # the database returns a distance
            "language": meta["language"],
            "title": meta["title"],
            "text": found["documents"][0][i][:60] + "...",
        })
    return pd.DataFrame(rows)

# An English question
display(search("Who copied books by hand?"))

In [ ]:
# An Arabic question: "Who translated the Greek books into Arabic?"
display(search("من ترجم الكتب اليونانية إلى العربية؟"))

Notice that the best results come in **both languages**. The question and the matching passage do not share the same words, and sometimes not even the same language. The database matches **meaning**.

## 5. Filtering with metadata

Often you want to search only part of the collection: one language, one topic, one period. The labels we stored make this easy.

In [ ]:
# Only Arabic texts
print("Arabic texts only:")
display(search("a famous university", where={"language": "ar"}))

# Two conditions together: English AND topic "travel and trade"
print("English texts about travel and trade:")
display(search("long journeys", where={"$and": [{"language": "en"}, {"topic": "travel and trade"}]}))

## 6. Adding new documents

A vector database grows easily. Let us add a passage about an Arabic newspaper of the Levant. First we search **before** adding it, then **after**.

In [ ]:
new_en = "The newspaper Filastin was published in Jaffa from 1911 and is now an important source for the history of the Levant."
new_ar = "صدرت جريدة «فلسطين» في يافا منذ عام 1911، وهي اليوم مصدر مهم لتاريخ بلاد الشام."

print("Before adding:")
display(search("Arabic newspapers of the Levant", k=2))

# Add the new passage in both languages, with its labels.
collection.add(
    ids=["p15_en", "p15_ar"],
    embeddings=embedder.encode([new_en, new_ar]).tolist(),
    documents=[new_en, new_ar],
    metadatas=[{"title": "Filastin newspaper", "topic": "archives and libraries", "language": "en"},
               {"title": "Filastin newspaper", "topic": "archives and libraries", "language": "ar"}],
)

print("After adding:")
display(search("Arabic newspapers of the Levant", k=2))

## 7. Seeing the results

The chart shows how close the six best results are to the question. Blue bars are English texts, orange bars are Arabic texts (each Arabic text is labelled with the English title of its passage).

In [ ]:
results = search("Who copied books by hand?", k=6)

labels = [f"{title} ({lang})" for title, lang in zip(results["title"], results["language"])]
colors = ["#2a6f97" if lang == "en" else "#e9a03b" for lang in results["language"]]

plt.figure(figsize=(7, 3.5))
# [::-1] reverses the lists so that the best result appears at the top
plt.barh(labels[::-1], results["similarity"].tolist()[::-1], color=colors[::-1])
plt.xlabel("similarity to the question (1 = same meaning)")
plt.title("Top 6 results: blue = English, orange = Arabic")
plt.tight_layout()
plt.show()

## Key takeaways

- A **vector database** stores vectors together with the original text and labels (metadata).
- It finds the **closest meaning** quickly, even with millions of items.
- **Metadata filters** let you search one language, topic or period.
- You can **add** new documents at any time.

**About saving:** our database lives in memory and disappears when the Colab session ends. To keep it on disk, use `chromadb.PersistentClient(path="my_database")` instead of `EphemeralClient()`. Files in Colab are also deleted when the session ends, so copy the folder to Google Drive if you need it later.

## Exercises

1. Add two passages from your own research area (in English or Arabic) and search for them.
2. Search for `"printing"` with the filter `{"topic": "writing and books"}`. Then remove the filter. What changes?
3. Ask the same question in English and in Arabic. Are the top results the same?

**Next:** Notebook 3 shows how to use a large language model through an API. Notebooks 4 and 5 then combine search and a language model to answer questions from documents (RAG).